In [ ]:
import marimo as mo

# SimulateFactory example
This notebook demonstrates how to configure `SimulateFactory` for Z⁰→μ⁺μ⁻ signal generations plus background processes,
then sketches the resulting invariant mass distributions for the prior, the smeared observations, and their difference.

In [ ]:
import matplotlib.pyplot as plt
import torch

from toymc_for_mnpe.generator import SimulateFactory, invariant_mass_from_ptphieta, mZ0

# create simulator factory
factory = SimulateFactory.create(device=torch.device('cpu'))

In [ ]:
# generate signal and background prior samples
signal_theta = torch.hstack([torch.full((16, 1), mZ0), torch.zeros((16, 1))])
background_theta_1 = torch.hstack([
    torch.linspace(70.0, 72.0, 4).unsqueeze(1),
    torch.ones((4, 1)),
])
background_theta_2 = torch.hstack([
    torch.linspace(115.0, 118.0, 4).unsqueeze(1),
    torch.ones((4, 1)),
])
theta = torch.vstack([signal_theta, background_theta_1, background_theta_2])

In [ ]:

events = factory.simulate(theta, generation_seed=123, smear_seed=321)
masses = invariant_mass_from_ptphieta(events)
delta = masses - theta[:, 0].unsqueeze(1)

print('batch shape:', events.shape)
print('prior labels:', theta[:, 1].unique())
print('first 5 reconstructed masses:', masses[:5].flatten())
print('delta mean:', delta.mean().item())

plt.close('all')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), tight_layout=True)
prior = theta[:, 0].numpy()
obs = masses.flatten().numpy()
delta_vals = delta.flatten().numpy()

axes[0].hist(prior, bins=20, color='tab:blue', alpha=0.7)
axes[0].set_title('Prior mass (theta[:,0])')
axes[0].set_xlabel('Mass / GeV')

axes[1].hist(obs, bins=20, color='tab:orange', alpha=0.7)
axes[1].set_title('Reconstructed mass from events smeared events')
axes[1].set_xlabel('Mass / GeV')

plt.show()